In [ ]:
# HMS - Harmful Brain Activity Classification
! pip install focal-loss-torch
from bisect import bisect_left
import copy
import os
import pickle
import subprocess
import sys
import time
from tqdm import tqdm
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed, ThreadPoolExecutor
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
import torch.utils.data as data
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, TensorDataset, DataLoader
import torch.optim as optim
from focal_loss.focal_loss import FocalLoss

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import roc_auc_score, precision_recall_curve, cohen_kappa_score, classification_report

# modeling_wav2vec2.py variables and imports
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2Processor

from IPython.display import FileLink, display

In [ ]:
config ={
    "use_ekg": False,
    "num_epochs": 10,
    "eeg_cols": [
         'Fp1', 'F3', 'C3', 'P3', 
         'F7', 'T3', 'T5', 'O1', 'Fz', 
         'Cz', 'Pz', 'Fp2', 'F4', 'C4', 
         'P4', 'F8', 'T4', 'T6', 'O2',
    ],
    
    #partitions
    "partition_config":{
        # "partition_name": "small_partition",
        "partition_name": "complete",
        "select_full_confidence_only": False,
        "sample_fraction": 1,
        "fc_min_fraction": 0.5,
        "uniform_weights": False,
    },
    
    #model
    "model_name": "facebook/wav2vec2-base",
    
    #criterion
    "criterion_name":"focal",
    "criterion_kwargs":{
        "gamma": 0.7,
    },
    
}

In [ ]:
BASE_DIR = "/kaggle/input/hms-harmful-brain-activity-classification/"
raw_meta_df_path = os.path.join(BASE_DIR, "train.csv")
df = pd.read_csv(raw_meta_df_path)

target_encoding = {'Seizure':0, 'LPD':1, 'GPD':2, 'LRDA':3, 'GRDA':4, 'Other':5,
                   'seizure':0, 'lpd':1, 'gpd':2, 'lrda':3, 'grda':4, 'other':5,
                   0:'seizure', 1:'lpd', 2:'gpd', 3:'lrda', 4:'grda', 5:'other'}
total_targets = 6

votes=['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
normed_votes_columns=[col+"_normed" for col in votes]
votes_map={k.split('_')[0]:i for i,k in enumerate(votes)}
votes_map.update({i:k.split('_')[0] for i,k in enumerate(votes)})

eeg_cols=config["eeg_cols"]
if config["use_ekg"]:
    eeg_cols.append("EKG")
num_channels=len(eeg_cols)

num_classes=6
sample_rate=200
sample_duration=50
batch_size=16
max_pool_size=1000
model_folder_path="/kaggle/input/model_0/transformers/default/2"
model_path="/kaggle/input/model_0/transformers/default/2/model_1.pkl"
dataset_folder_path="/kaggle/input/model_0/transformers/default/2"
df['total_votes'] = df['seizure_vote']+df['lpd_vote']+df['gpd_vote']+df['lrda_vote']+df['grda_vote']+df['other_vote']
df['seizure_vote_normed'] = df['seizure_vote']/df['total_votes']
df['lpd_vote_normed'] = df['lpd_vote']/df['total_votes']
df['gpd_vote_normed'] = df['gpd_vote']/df['total_votes']
df['lrda_vote_normed'] = df['lrda_vote']/df['total_votes']
df['grda_vote_normed'] = df['grda_vote']/df['total_votes']
df['other_vote_normed'] = df['other_vote']/df['total_votes']

In [ ]:
def get_col_ix(col_id): #Consistency checked
    if isinstance(col_id, int):
        return 9+col_id
    elif isinstance(col_id, str):
        return 9+target_encoding[col_id]
    
def get_vote_col(col_id): #Consistency checked
    if isinstance(col_id, int):
        return f'{target_encoding[col_id]}_vote'
    elif isinstance(col_id, str):
        return f'{col_id}_vote'
    
def get_normed_ix(col_id): #Consistency checked
    if isinstance(col_id, int):
        return 16+col_id
    elif isinstance(col_id, str):
        return 16+target_encoding[col_id]
    
def get_normed_col(col_id): #Consistency checked
    if isinstance(col_id, int):
        return f'{target_encoding[col_id]}_vote_normed'
    elif isinstance(col_id, str):
        return f'{col_id}_vote_normed'
def get_eeg_path(id):
    eeg_path=eeg_path = f"/kaggle/input/hms-harmful-brain-activity-classification/train_eegs/{id}.parquet"
    return eeg_path

def get_eeg(id):
    eeg_path = f"/kaggle/input/hms-harmful-brain-activity-classification/train_eegs/{id}.parquet"
    eeg = pd.read_parquet(eeg_path)[eeg_cols]
    return eeg

def get_eeg_sample(ix, meta_df, return_normalized_votes=False):
    id=meta_df.loc[ix, 'eeg_id']
    eeg=get_eeg(id)
    start_frame=int(meta_df.loc[ix, 'eeg_label_offset_seconds']*200)
    end_frame=start_frame+10000
    eeg_sample=torch.from_numpy(eeg.iloc[start_frame: end_frame].to_numpy())
    if not return_normalized_votes:
        return eeg_sample
    normalized_votes = torch.from_numpy(meta_df.loc[ix, normed_votes_columns].to_numpy(dtype=np.float32))
    return normalized_votes, eeg_sample

def download_file(path, download_file_name):
    os.chdir('./')
    zip_name = f"/kaggle/working/{download_file_name}.zip"
    command = f"zip {zip_name} {path} -r"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("Unable to run zip command!")
        print(result.stderr)
        return 
    display(FileLink(f'{download_file_name}.zip'))

def eeg_sample_isnan(ix, meta_df):
    eeg_sample = get_eeg_sample(ix, meta_df).transpose(0, 1)
    scaler = StandardScaler()
    eeg_sample = scaler.fit_transform(eeg_sample.T).T
    if np.isnan(eeg_sample).any():
        return ix
    return None
    
def prune_nans(meta_df):
    total_unpruned_rows = meta_df.shape[0]
    nan_indices = set()
    
    with ThreadPoolExecutor(max_workers=4) as executor:
        results = list(tqdm(executor.map(lambda idx: eeg_sample_isnan(idx, meta_df), range(total_unpruned_rows)),total=total_unpruned_rows))
    
    nan_indices = set(filter(None, results))
    ans_df=meta_df.loc[meta_df.index[~meta_df.index.isin(nan_indices)]].reset_index()
    return ans_df

In [ ]:
df['confidence'] = df.apply(lambda row: row[get_normed_col(target_encoding[row['expert_consensus']])], axis=1)
df['full_confidence'] = (df['confidence'] > 0.999) & (df['confidence'] < 1.001)

bins = [i / 10 for i in range(11)]
bins[-1]+=0.01 
labels = [(i+0.5)/10 for i in range(10)]
df['confidence_classes'] = pd.cut(df['confidence'], bins=bins, labels=labels, include_lowest=True)

meta_fc_df=df[df['full_confidence']]
meta_fc_df.drop(columns=['full_confidence', 'confidence_classes'], inplace=True)
meta_df = copy.deepcopy(df)
sample_fraction = config["partition_config"]["sample_fraction"]
fc_min_fraction = config["partition_config"]["fc_min_fraction"]
uniform_weights = config["partition_config"]["uniform_weights"]
select_full_confidence_only = config["partition_config"]["select_full_confidence_only"]

if select_full_confidence_only:
    meta_df = meta_fc_df.sample(frac=sample_fraction).reset_index(drop=True)
else:
    meta_df = meta_df.sample(frac=sample_fraction).reset_index(drop=True)

all_class_dfs=[]
for vote in votes:
    all_class_dfs.append(meta_df[meta_df[vote]>0.01])
counts=[class_df.shape[0] for class_df in all_class_dfs]
total_count=meta_df.shape[0]
weights = torch.tensor([count/total_count for count in counts])
    
if uniform_weights:
    min_count=min(counts)
    print(min_count)
    len_samples=int(min_count*fc_fraction)
    all_samples=[sample_df.sample(n=len_samples, replace=True) for i,sample_df in enumerate(all_class_dfs)]
    meta_df = pd.concat(all_samples, ignore_index=True)
    weights = torch.tensor([1 for i in range(num_classes)])
total_count=meta_df.shape[0]
if config["partition_config"]["partition_name"]=="small_partition":
    meta_df=meta_df.sample(frac=0.001, ignore_index=True)
    
meta_df=prune_nans(meta_df)
print(f"meta_df.shape = ", meta_df.shape)

In [ ]:
_HIDDEN_STATES_START_POSITION = 2
class ModelLoader(Wav2Vec2ForSequenceClassification):
    loss_fct = None
    @classmethod
    def from_pretrained(
        cls,
        pretrained_model_name_or_path,
        *model_args,
        config = None,
        cache_dir = None,
        ignore_mismatched_sizes = False,
        force_download = False,
        local_files_only = False,
        token = None,
        revision = "main",
        use_safetensors = None,
        # weights_only = True,
        criterion = None,
        **kwargs,
    ):
        if criterion is None:
            cls.loss_fct = CrossEntropyLoss()
        else:
            cls.loss_fct = criterion
        return Wav2Vec2ForSequenceClassification.from_pretrained(
            pretrained_model_name_or_path,
            *model_args,
            config = None,
            cache_dir = None,
            ignore_mismatched_sizes = False,
            force_download = False,
            local_files_only = False,
            token = None,
            revision = "main",
            use_safetensors = None,
            # weights_only = True,
            **kwargs,
        )

    def forward(
        self,
        input_values,
        attention_mask = None,
        output_attentions = None,
        output_hidden_states = None,
        return_dict = None,
        labels = None,
    ):
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """

        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        output_hidden_states = True if self.config.use_weighted_layer_sum else output_hidden_states

        outputs = self.wav2vec2(
            input_values,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        if self.config.use_weighted_layer_sum:
            hidden_states = outputs[_HIDDEN_STATES_START_POSITION]
            hidden_states = torch.stack(hidden_states, dim=1)
            norm_weights = nn.functional.softmax(self.layer_weights, dim=-1)
            hidden_states = (hidden_states * norm_weights.view(-1, 1, 1)).sum(dim=1)
        else:
            hidden_states = outputs[0]

        hidden_states = self.projector(hidden_states)
        if attention_mask is None:
            pooled_output = hidden_states.mean(dim=1)
        else:
            padding_mask = self._get_feature_vector_attention_mask(hidden_states.shape[1], attention_mask)
            expand_padding_mask = padding_mask.unsqueeze(-1).repeat(1, 1, hidden_states.shape[2])
            hidden_states[~expand_padding_mask] = 0.0
            pooled_output = hidden_states.sum(dim=1) / padding_mask.sum(dim=1).view(-1, 1)

        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            # loss_fct = CrossEntropyLoss()
            loss = ModelLoader.loss_fct(logits.view(-1, self.config.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[_HIDDEN_STATES_START_POSITION:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )
                

In [ ]:
# Model and Processor

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

criterion=None
if config["criterion_name"] == "focal":
    criterion=FocalLoss(weights=weights, **config["criterion_kwargs"])
elif config["criterion_name"] == "crossentropy":
    criterion=CrossEntropyLoss(weights=weights, **config["criterion_kwargs"])
elif config["criterion_name"] == "focal_vat":
    criterion=CrossEntropyLoss(weights=weights, **config["criterion_kwargs"])
else:
    raise NotImplementedError(f"Criterion not implemented for {config['criterion_name']}. Only for focal, crossentropy, focal_vat")

model = ModelLoader.from_pretrained(
    pretrained_model_name_or_path = "facebook/wav2vec2-base",
    num_labels = num_classes,
    criterion = FocalLoss(gamma=0.7, weights=torch.tensor(weights)),
)

if os.path.exists(model_folder_path):
    print("Saved model loaded")
    model = ModelLoader.from_pretrained(
    pretrained_model_name_or_path = model_path,
    num_labels=num_classes
    )    

# Freeze feature extractor layers
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False

time.sleep(3)

In [ ]:
# Splitting and creating partitions
if (not os.path.isdir(dataset_folder_path)):
    if not config["partition_config"]["partition_name"]=="patient_agnostic":
        len_meta_df=meta_df.shape[0]
        test_frac=0.15
        val_frac=0.15
        train_meta_df=meta_df.iloc[:int((1-test_frac-val_frac)*len_meta_df),:]
        val_meta_df=meta_df.iloc[int((1-test_frac-val_frac)*len_meta_df):int((1-test_frac)*len_meta_df),:]
        test_meta_df=meta_df.iloc[int((1-test_frac)*len_meta_df):, :]
        pool_size = int((max_pool_size//batch_size)*batch_size)
    else:
        patient_group=meta_df.groupby("patient_id", as_index=False).agg({"eeg_id":"count"})
        patient_group.sample(frac=1).reset_index(drop=True)
        patient_group.rename(columns={"eeg_id":"count"}, inplace=True)
        patient_group["cumulative_count"]=patient_group["count"].cumsum()
        
        len_meta_df=meta_df.shape[0]
        train_samples=(1-test_fraction-validation_fraction)*len_meta_df
        last_train_patient=patient_group['cumulative_count'].searchsorted(train_samples, side='right')
        train_meta_df=meta_df[meta_df['patient_id'].isin(patient_group.loc[:last_train_patient, 'patient_id'])]
        
        validation_samples=validation_fraction*len_meta_df
        last_validation_patient=patient_group['cumulative_count'].searchsorted(train_samples+validation_samples, side='right')
        val_meta_df=meta_df[meta_df['patient_id'].isin(patient_group.loc[last_train_patient+1:last_validation_patient, 'patient_id'])]
        
        last_test_patient=patient_group['cumulative_count'].searchsorted(len_meta_df, side='right')
        test_meta_df=meta_df[meta_df['patient_id'].isin(patient_group.loc[last_validation_patient+1:last_test_patient, 'patient_id'])]

    train_meta_df.to_csv("train_meta_df.csv")
    test_meta_df.to_csv("test_meta_df.csv")
    val_meta_df.to_csv("val_meta_df.csv")
    
else:
    train_meta_path=dataset_folder_path+"/train_meta_df.csv"
    val_meta_path=dataset_folder_path+"/val_meta_df.csv"
    test_meta_path=dataset_folder_path+"/test_meta_df.csv"

    test_meta_df=pd.read_csv(test_meta_path)
    train_meta_df=pd.read_csv(train_meta_path)
    val_meta_df=pd.read_csv(val_meta_path)

In [ ]:
# Custom Dataset Class
class EEGDataset(Dataset):
    def __init__(self, meta_df, pool_size, name, processor=processor, target_sampling_rate=16000, orig_sampling_rate=200):
        # super().__init__(meta_df, pool_size, name)
        self.pool_size = pool_size
        self.meta_df = meta_df
        self.name = name
        self.processor = processor
        self.orig_sampling_rate = orig_sampling_rate
        self.target_sampling_rate = target_sampling_rate
        
        self.scaler = StandardScaler()
        print(f"EEGDataset {name} initialized\n")
        
    def __len__(self):
        return self.meta_df.shape[0]
    
    def __getitem__(self, idx):
        ix=idx
        normalized_vote, eeg_sample = get_eeg_sample(ix, self.meta_df, True)
        eeg_sample = eeg_sample.transpose(0, 1)
        eeg_sample = self.scaler.fit_transform(eeg_sample.T).T
        label = votes_map[self.meta_df.loc[ix, 'expert_consensus'].lower()]
        inputs = self.processor(eeg_sample, sampling_rate=self.target_sampling_rate, return_tensors="pt", padding=True)
        inputs["labels"] = torch.tensor(label, dtype=torch.long)
        inputs["normalized_votes"] = normalized_vote
        return inputs
      
pool_size=(max_pool_size//batch_size)*batch_size

train_meta_df.reset_index(drop=True, inplace=True)
val_meta_df.reset_index(drop=True, inplace=True)
test_meta_df.reset_index(drop=True, inplace=True)

if (not os.path.isdir(dataset_folder_path)):
    train_dataset = EEGDataset(train_meta_df, pool_size, "train")
    val_dataset = EEGDataset(val_meta_df, pool_size, "val")
    test_dataset = EEGDataset(test_meta_df, pool_size, "test")
else:
    train_dataset=EEGDataset(train_meta_df, pool_size, "train")
    test_dataset=EEGDataset(test_meta_df, pool_size, "test")
    val_dataset=EEGDataset(val_meta_df, pool_size, "val")

train_loader = DataLoader(train_dataset, batch_size=batch_size, prefetch_factor=1, pin_memory=True, num_workers=1)
val_loader = DataLoader(val_dataset, batch_size=batch_size, prefetch_factor=1, pin_memory=True, num_workers=1)
test_loader = DataLoader(test_dataset, batch_size=batch_size, prefetch_factor=1, pin_memory=True, num_workers=1)
# Optimizer and Loss
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
num_epochs=config["num_epochs"]
epoch_offset=0

In [ ]:
class EvaluatePartition:
    def __init__(self, model, data_loader, name, device=device):
        self.model=model
        self.data_loader=data_loader
        self.name=name
        self.device=device

        self.calc_softmax=torch.nn.Softmax(dim=-1)  
        self.data_preds = []
        self.data_labels = []
        self.data_probs = []
        self.total_loss = 0.0
        self.total_kl_loss = 0.0
        self.num_classes = None
        self.final_metrics = {}
        

    def _reset_base_metrics(self, base_metrics=None):
        self.data_preds = []
        self.data_labels = []
        self.data_probs = []
        self.total_loss = 0.0
        self.total_kl_loss = 0.0
        if base_metrics is not None:
            self.data_preds = base_metrics["data_preds"]
            self.data_labels = base_metrics["data_labels"]
            self.data_probs = base_metrics["data_probs"]
            self.total_loss = base_metrics["total_loss"]
            self.total_kl_loss = base_metrics["total_kl_loss"]
        self.final_metrics.clear()
    
    def _evaluate(self, base_metrics=None):
        self._reset_base_metrics(base_metrics)
        
        if base_metrics is None:
            with torch.no_grad():
                for batch in tqdm(self.data_loader, desc=f"Checking model on {self.name} dataset", ncols=100, leave=True, dynamic_ncols=True):
                    inputs=self._get_input_kwargs(batch)
                    normalized_votes=inputs['normalized_votes']
                    del inputs['normalized_votes']
                    
                    outputs = self.model(**inputs)
                    
                    inputs['normalized_votes']=normalized_votes
                    loss=outputs.loss
                    self.total_loss+=loss.item()
                    logits, preds, probs, labels = self._process_model_outputs(inputs, outputs)
                
        return self._get_final_metrics()
    
    def _get_input_kwargs(self, batch):
        inputs = {}
        inputs['input_values'] = batch['input_values'].to(torch.float32).to(self.device)
        if len(inputs['input_values'].shape) == 1:
            inputs['input_values'] = torch.unsqueeze(inputs['input_values'], 0)
            inputs['input_values'].to(self.device)
        inputs['input_values'] = torch.reshape(inputs['input_values'], (-1, num_channels*10000))
        inputs['labels'] = batch['labels'].to(torch.int64).to(self.device)
        inputs['normalized_votes']=batch['normalized_votes'].to(self.device)
        return inputs

    def _process_model_outputs(self, inputs, outputs):
        logits = outputs.logits 
        # Collect predictions and labels
        preds = torch.argmax(logits, dim=-1)
        self.data_preds.extend(preds.tolist())

        labels = inputs['labels']
        self.data_labels.extend(labels.tolist())
        
        probs = self.calc_softmax(logits)
        self.data_probs.append(probs.detach().cpu().numpy())

        # print("probabilities = ", F.log_softmax(logits, dim=-1))
        # print("log probabilities = ", torch.log(F.log_softmax(logits, dim=-1)))
        kl_loss = F.kl_div(F.log_softmax(logits, dim=-1), inputs['normalized_votes'], reduction='batchmean')
        self.total_kl_loss += kl_loss.item()
        
        return logits, preds, probs, labels

    def _get_final_metrics(self, base_metrics=None):
        self._finalize_epoch_data()
        self.num_classes = self.data_probs.shape[1]
        self._add_regressive_scores()
        self._add_evaluation_graphs()
        return self.final_metrics
        
    def _finalize_epoch_data(self):
        self.data_labels = torch.tensor(self.data_labels).cpu().numpy()
        self.data_preds = torch.tensor(self.data_preds).cpu().numpy()
        if isinstance(self.data_probs, list):
            self.data_probs = np.concatenate(self.data_probs, axis=0)
        self.data_probs = torch.tensor(self.data_probs).cpu().numpy()
        
    def _add_regressive_scores(self):
        data_accuracy = accuracy_score(self.data_labels, self.data_preds)
        data_f1 = f1_score(self.data_labels, self.data_preds, average='weighted')
        data_precision = precision_score(self.data_labels, self.data_preds, average='weighted')
        data_recall = recall_score(self.data_labels, self.data_preds, average='weighted')
        cohen_kappa = cohen_kappa_score(self.data_labels, self.data_preds)
        
        regressive_scores={
            "accuracy": data_accuracy,
            "f1_score": data_f1,
            "precision": data_precision,
            "recall": data_recall,
            "cohen_kappa": cohen_kappa,
            "total_loss": self.total_loss,
            "total_kl_loss": self.total_kl_loss,
        }
        print(f"{self.name} Dataset - Accuracy: {data_accuracy:.4f} | Precision: {data_precision:.4f} | Recall: {data_recall:.4f} | F1: {data_f1:.4f}")
        print(f"Total Loss: {self.total_loss:.4f} | Total KL Loss: {self.total_kl_loss:.4f} | cohen_kappa: {cohen_kappa:.4f}")
        self.final_metrics.update(regressive_scores)
        
    def _add_evaluation_graphs(self):
        try:
            auc_roc = roc_auc_score(self.data_labels, self.data_probs, multi_class='ovr', average='weighted')
        except ValueError:
            auc_roc = None
    
        pr_curves = {}
        for i in range(self.num_classes):
            precision, recall, _ = precision_recall_curve(self.data_labels == i, self.data_probs[:, i])
            pr_curves[f"class_{i}"] = {"precision": precision.tolist(), "recall": recall.tolist()}
        class_report = classification_report(self.data_labels, self.data_preds, output_dict=True)
        
        evaluation_curves={
            "auc_roc": auc_roc,
            "support": class_report["weighted avg"]["support"],
            "pr_curves": pr_curves
        }
        self.final_metrics.update(evaluation_curves)


In [ ]:
# # Training Loop - EvaluatePartition based
experiment_data={}
experiment_data["experiment_config"] = config

training = EvaluatePartition(model = model, data_loader = train_loader, name="train", device=device)
validation = EvaluatePartition(model = model, data_loader = val_loader, name="val", device=device)
testing = EvaluatePartition(model = model, data_loader = test_loader, name="test", device=device)

for epoch_id in range(num_epochs):
    epoch=epoch_id+epoch_offset
    model.train()
    total_loss = 0.0
    total_kl_loss = 0.0
    train_preds, train_labels, train_probs = [], [], []

    training._reset_base_metrics()
    print(f"For Epoch: {epoch}")
    epoch_data={
        f"{training.name}_metrics":{},
        f"{validation.name}_metrics":{},
        f"{testing.name}_metrics":{},
    }
    for i, batch in enumerate(tqdm(train_loader)):
        inputs = training._get_input_kwargs(batch)
        normalized_votes=inputs['normalized_votes']
        del inputs['normalized_votes']
        
        outputs = model(**inputs)
        inputs['normalized_votes']=normalized_votes
        
        loss = outputs.loss
        training.total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()        

        training._process_model_outputs(inputs, outputs)
        

    train_metrics = training._get_final_metrics()
    val_metrics = validation._evaluate()
    test_metrics = testing._evaluate()

    epoch_data[f"{training.name}_metrics"].update(train_metrics)
    epoch_data[f"{validation.name}_metrics"].update(val_metrics)
    epoch_data[f"{testing.name}_metrics"].update(test_metrics)
    experiment_data[f"epoch_{epoch}"] = {}
    experiment_data[f"epoch_{epoch}"].update(epoch_data)
    
    model.save_pretrained(f"model_{epoch}")
    # download_file(f"./model_{epoch}.pkl", f"model_{epoch}.pkl")
    print("Model saved")
    
    print()

final_data_path = "./experiment_data.pkl"

with open(final_data_path, 'wb') as f:
    pickle.dump(experiment_data, f)
print(os.listdir("/kaggle/working/"))
